## Data Ingestion to Vector DB Pipeline


In [16]:
import os
from google.colab import drive

# Colab setup: install dependencies and mount Google Drive.
!pip install -q langchain langchain-community langchain_text_splitters sentence-transformers chromadb scikit-learn torch python-dotenv

drive.mount('/content/drive')

# Adjust this if your repo is located in a different Drive folder.
base_path = '/content/drive/MyDrive/RAG SCRIPTS'
notebook_path = os.path.join(base_path, 'notebook')
os.chdir(notebook_path)

print('Mounted Google Drive')
print('Base path:', base_path)
print('Notebook path:', notebook_path)
print('Working directory:', os.getcwd())

ModuleNotFoundError: No module named 'google.colab'

#### Data Ingestion

In [ ]:
"""
Data ingestion dependencies for the PDF-based RAG pipeline.

The pipeline uses PyPDFLoader to read PDF pages as documents,
RecursiveCharacterTextSplitter to break long text into overlapping chunks,
and Path to work with file and folder paths in a platform-independent way.
"""
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Brooklyn\AppData\Local\Temp\ipykernel_24856\1221245361.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\Brooklyn\OneDrive\Desktop\RAG SCRIPTS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def process_all_pdfs(pdf_directory):
    """
    Load all PDF files from a directory and its subdirectories.

    This function scans the given directory recursively for PDF files, loads
    each file page by page using `PyPDFLoader`, and returns a flat list of
    LangChain Document objects. It also enriches each document's metadata with
    the source filename and file type so the information can be preserved during
    downstream chunking, embedding, and retrieval steps in the RAG pipeline.

    Args:
        pdf_directory (str): Path to the root folder containing PDF files.

    Returns:
        list: A flattended list of LangChain Document objects loaded from all discovered PDFs.

    Notes:
        - Each PDF is processed independently.
        - If one file fails to load, the function logs the error and continues
          processing the remaining files.
        - Metadata fields added:
            - source_file: Original PDF filename
            - file_type: Always set to "pdf"
    """
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process") 
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
           
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents) 
            print(f"  ✓ Loaded {len(documents)} pages") 
            
        except Exception as e: 
            print(f"  ✗ Error: {e}")


    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Embeddings.pdf
  ✓ Loaded 23 pages

Processing: proceedingbook-Anas_Mustafa.pdf
  ✓ Loaded 9 pages

Processing: RAG_Paper___Bhavith_and_Preethika.pdf
  ✓ Loaded 12 pages

Total documents loaded: 44


#### Chunking

In [ ]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split LangChain documents into smaller overlapping chunks for retrieval.

    This function uses `RecursiveCharacterTextSplitter` to break large document
    pages into smaller chunks that are easier to embed and retrieve in a RAG
    pipeline. Chunk overlap is used to preserve context across chunk boundaries,
    which helps reduce information loss when relevant text spans multiple chunks.

    Args:
        documents (list): List of LangChain Document objects.
        chunk_size (int, optional): Maximum size of each chunk in characters.
            Defaults to 1000.
        chunk_overlap (int, optional): Number of overlapping characters between
            consecutive chunks. Defaults to 200.

    Returns:
        list: A list of chunked LangChain Document objects.

    Notes:
        - Splitting is done recursively using the separators:
          `["\\n\\n", "\\n", " ", ""]`
        - Text length is measured using raw character count via `len`.
        - Original metadata is preserved on the output chunks.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
   
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
## Chunking documents
chunks=split_documents(all_pdf_documents)
chunks

Split 44 documents into 220 chunks

Example chunk:
Content: Published as a conference paper at ICLR 2026
ON THETHEORETICALLIMITATIONS OF
EMBEDDING-BASEDRETRIEVAL
Orion Weller1,2 Michael Boratko1 Iftekhar Naim1 Jinhyuk Lee1
1Google DeepMind, 2Johns Hopkins Univ...
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:8e18bb9)', 'creationdate': '', 'author': 'Orion Weller; Michael Boratko; Iftekhar Naim; Jinhyuk Lee', 'doi': 'https://doi.org/10.48550/arXiv.2508.21038', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'On the Theoretical Limitations of Embedding-Based Retrieval', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2508.21038v2', 'source': '..\\data\\pdf\\Embeddings.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Embeddings.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:8e18bb9)', 'creationdate': '', 'author': 'Orion Weller; Michael Boratko; Iftekhar Naim; Jinhyuk Lee', 'doi': 'https://doi.org/10.48550/arXiv.2508.21038', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'On the Theoretical Limitations of Embedding-Based Retrieval', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2508.21038v2', 'source': '..\\data\\pdf\\Embeddings.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Embeddings.pdf', 'file_type': 'pdf'}, page_content='Published as a conference paper at ICLR 2026\nON THETHEORETICALLIMITATIONS OF\nEMBEDDING-BASEDRETRIEVAL\nOrion Weller1,2 Michael Boratko1 Iftekhar Naim1 Jinhyuk Lee1\n1Google DeepMind, 2Johns Hopkins University\noweller@cs.jhu.edu,jinhyuklee@google.com\nABSTRACT\nVector embeddings have been t

#### Embedding 



In [ ]:
# Sets up libraries used for embedding generation, vector storage with Chroma, typed helper methods, unique document IDs, and similarity-based retrieval.
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
#  Wrapper for loading a SentenceTransformer model and generating embeddings.
class EmbeddingManager:
    """
    Load and manage a SentenceTransformer embedding model.

    This helper class initializes a sentence-transformer model once and reuses it
    to generate dense vector embeddings for input texts. It is designed for RAG
    pipelines where document chunks and user queries must be converted into
    numerical representations for similarity search.
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager and load the model.

        Args:
            model_name (str, optional): Name of the SentenceTransformer model.
                Defaults to "all-MiniLM-L6-v2".
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """
        Load the SentenceTransformer model into memory.

        Raises:
            Exception: Re-raises any model-loading error after logging it.
        """
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Encode a list of texts into dense vector embeddings.

        Args:
            texts (List[str]): Input texts to encode.
            show_progress_bar (bool, optional): Whether to display encoding progress.

        Returns:
            np.ndarray: Embedding matrix.
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5933.11it/s]


Model loaded successfully. Embedding dimension: 384


#### VectorStore DB

In [ ]:
class VectorStore:
    """
    Wrapper around ChromaDB for storing document embeddings and metadata.

    This class initializes a persistent Chroma collection and provides a helper
    method for adding LangChain Document chunks together with their embeddings,
    text content, and metadata for later semantic retrieval.
    """
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """        
        Initialize the vector store.

        Args:
            collection_name (str, optional): Name of the Chroma collection.
            persist_directory (str, optional): Directory for persistent storage.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """
        Initialize the persistent Chroma vector store.

        This method ensures that the persistence directory exists, creates a
        `chromadb.PersistentClient` pointing to that directory, and retrieves or
        creates a collection with the configured `collection_name`. It also logs
        basic information about the initialized collection, including the current
        document count.

        Raises:
        Exception: Any error during client or collection initialization is
        logged and re-raised.
    """
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the Chroma collection.

        This method expects a list of LangChain `Document`-like objects and a
        NumPy array of embeddings aligned by index. For each document embedding
        pair it generates a unique ID, augments the document metadata with
        `doc_index` and `content_length`, and collects the page content and
        embedding as plain Python types suitable for Chroma. All items are
        then inserted into the underlying collection in a single `add` call.

        Args:
            documents (List[Any]): Sequence of documents to store, each expected
            to have `.metadata` and `.page_content` attributes.
        embeddings (np.ndarray): 2D array of embedding vectors, one row per
            document.

        Raises:
            ValueError: If the number of documents does not match the number
            of embeddings.
        Exception: Any error from the Chroma `add` operation is logged and
            re-raised.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
      
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 2200


#### Convert the Text to embeddings
##### This section demonstrates the core RAG steps of taking chunked text, generating embeddings, and storing everything in the vector store.

In [ ]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 220 texts...


Batches: 100%|██████████| 7/7 [00:08<00:00,  1.24s/it]


Generated embeddings with shape: (220, 384)
Adding 220 documents to vector store...
Successfully added 220 documents to vector store
Total documents in collection: 2420


 #### To check that everything lines up!

In [ ]:
len(chunks), len(texts), len(embeddings)
type(embeddings), embeddings.shape 
# 220 = number of rows → We have 220 chunks / texts.
# 384 = embedding dimension → each text is represented by a 384‑dimensional vector.

(numpy.ndarray, (220, 384))

## Retriever Pipeline - From the VectorStore


In [ ]:

class RAGRetriever:
    """Retrieve relevant documents from the vector store for a user query.

    The retriever embeds the query, searches the vector store for nearest
    neighbors, and returns ranked results with content, metadata, distance,
    and a derived similarity score.
    """
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriever with a vector store and embedding manager.

        Args:
            vector_store: Store containing indexed document embeddings.
            embedding_manager: Component used to generate query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Return the most relevant documents for a query.

        Args:
            query: Natural-language search query.
            top_k: Maximum number of results to request from the vector store.
            score_threshold: Minimum similarity score required to keep a result.

        Returns:
            A list of ranked retrieval results. Each result contains:
            - id: Document identifier in the vector store.
            - content: Retrieved chunk text.
            - metadata: Stored metadata for the chunk.
            - similarity_score: Derived score where higher is better.
            - distance: Raw distance returned by the vector store.
            - rank: 1-based rank in the returned results.
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
       
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], 
                n_results=top_k 
            )
            
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
             
             
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold: # if the similarity score meets the threshold, we include this document in the context document which is retrieved_docs
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [ ]:
# A check on the retreival function
rag_retriever.retrieve("Neural Embedding models")

Retrieving documents for query: 'Neural Embedding models'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.57it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3289e33d_8',
  'content': 'impossible for models with small embedding dimensions using standard optimization techniques.\nOverall, our work contributes: (1) a theoretical basis for the fundamental limitations of embedding\nmodels, (2) a best-case empirical analysis showing that this proof holds for any dataset instantiation\n(by free embedding optimization), and (3) a simple real-world natural language instantiation called\nLIMIT that even state-of-the-art embedding models cannot solve.\nThese results imply interesting findings for the community: on one hand we see neural embedding\nmodels becoming immensely successful. However, academic benchmarks test only a small amount\nof the queries that could be issued (and these queries are often overfitted to), hiding these limitations.\nOur work shows that as the tasks given to embedding models require returning ever-increasing\ncombinations of top-k relevant documents (e.g., through instructions connecting previously unrelated',

## RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# Initialize the Groq LLM (requires GROQ_API_KEY in the environment).
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

def rag_simple(query, retriever, llm, top_k=3):
    """Run a simple RAG step: retrieve context and generate an answer.

    Args:
        query: User question string.
        retriever: RAGRetriever instance used to fetch relevant chunks.
        llm: ChatGroq LLM instance used to generate the answer.
        top_k: Number of retrieved chunks to use as context.

    Returns:
        Model-generated answer, or a fallback message if no context is found.
    """
    # Retrieve the most relevant chunks for the query
    results = retriever.retrieve(query, top_k=top_k) 
   
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    
    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.

        Context:
        {context}

        Question: {query}

        Answer:"""
    
    # Call the LLM and return the answer text.
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content



In [ ]:
# Returns the final answer as plain text.
answer = rag_simple("What are the advantages of using FAISS for similarity search?", rag_retriever, llm)
print("Answer:", answer)

Retrieving documents for query: 'What are the advantages of using FAISS for similarity search?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.19it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The advantages of using FAISS for similarity search include:

1. **Fast and efficient similarity searches**: FAISS is designed for real-time retrieval in large-scale datasets.
2. **Reduced search space**: FAISS creates an index that partitions the embedding space into smaller regions, reducing the search space for each query.
3. **Implementation of nearest neighbor search algorithms and indexing techniques**: FAISS supports various techniques such as Product Quantization (PQ) and Inverted File Indexing (IVF) for efficient similarity search.
4. **Optimized search process**: FAISS optimizes the search process by reducing the computational complexity of similarity searches.


### Introduction of Advanced RAG Pipeline Features - Quality Filtering, Safe Fallback, Traceability, and Signals for Evaluation
 ###### 1.  Score threshold and top‑k control is now passed a) we cap the number of chunks used (top_k=5) b) we filter out low‑similarity chunks using min_score
###### (“normal” simple RAG often just does “top_k=3” without a threshold, so even badly matched docs can still go into the prompt and increase hallucinations. )
 ######  This code gives you better control over retrieval quality instead of blindly trusting whatever comes back.
###### 2. Explicit “no context” fallback - This is an advanced safety behavior: our app can detect “no grounding” and handle it (e.g., ask user to rephrase, fall back to web search, etc.)
###### 3. Structured source metadata for each chunk - source, page number if available, similarity score of each chunk, preview of first 300 chars of the content.
###### (It enables:UI with clickable citations and previews, Debugging,Logging for evaluation and RAG analysis)
###### 4. Simple confidence estimate - Introduced a scalar “confidence” score derived from retrieval similarity. Apps can show confidence bar, route low‑confidence cases to a human loop, log and evaluate performance by confidence bins.
###### 5. Intoduced Optional full context return - Controlled by return_context: If True, we attach the full concatenated context to the output; if False, we keep the payload small. (This is useful when: backend just needs answer + sources and admin/debug tools need to see exactly what was fed to the LLM. ). A simple RAG often only gives back the answer, forcing us to change code if we later want to inspect context.
###### 6. Answer + rich metadata as a single object -  It pushes the code closer to a production RAG API which is debuggable, explainable, extensible. 

In [ ]:
from torch import empty

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """Retrieve context and generate an answer with source details.

    Args:
        query: User question.
        retriever: Object with a retrieve() method.
        llm: Language model object with an invoke() method.
        top_k: Number of top documents to retrieve.
        min_score: Minimum similarity score for filtering results.
        return_context: Whether to include the full context in the output.

    Returns:
        A dictionary containing the answer, sources, confidence, and
        optionally the full retrieved context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context = "\n\n".join([doc['content'] for doc in results]) 
    
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])
    
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,   
        'sources': sources,           
        'confidence': confidence      
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Text Chunking Algorithm", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Text Chunking Algorithm'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.20it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The Text Chunking Algorithm is outlined in Algorithm 1. 

Algorithm 1:
1. Initialize text as the raw text extracted from the PDF
2. Initialize chunk size ← 1000
3. Initialize chunk overlap ← 200
4. for each page in text do chunks ← split(page, chunk size, chunk overlap)
5. for each chunk in chunks do
6. if chunk boundary cuts a sentence then
7. Adjust boundary to nearest sentence delimiter
8. end if
9. end for
10. end for
11. return chunks
Sources: [{'source': 'RAG_Paper___Bhavith_and_Preethika.pdf', 'page': 4, 'score': 0.4310457706451416, 'preview': 'coreference issues in downstream tasks.\n– Boundary Adjustment : The splitter intelligently\nadjusts chunk boundaries to avoid cutting sentences\nor paragraphs inappropriately. This involves back-\ntracking to the nearest sentence delimiter if the chunk\nsize limit is exceeded mid-sentence.\n– Chunk Index...'}, {'source': 'RAG_Paper___Bhavith_and_Preethika.pdf', 'page': 4, 'score': 0.4310457706451416, 'preview': 'coreference issue

### Advanced RAG Pipeline: We add Streaming, Citations, History, Summarization!
##### This class is basically turning our earlier RAG function into a reusable, stateful “pipeline object” with some extra UX features: streaming, inline citations, summaries, and history

In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    """Simple RAG pipeline with retrieval, answering, optional summary, and history."""
    def __init__(self, retriever, llm):
        """Initialize the pipeline with a retriever and language model."""
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        """Answer a question using retrieved context.

        Args:
            question: User query.
            top_k: Number of top documents to retrieve.
            min_score: Minimum similarity score for retrieval.
            stream: Whether to simulate streaming output.
            summarize: Whether to generate a short summary.

        Returns:
            A dictionary with the question, answer, sources, summary, and history.
        """
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score) 
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results]) # Builds the context string for the LLM if we have results.
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)] 
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer
        # Optionally generate a short summary
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # History tracking - makes it easy to inspect previous interactions later.
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("representational capacity of vector embeddings", top_k=5, min_score=0.2, stream=True, summarize=True) 
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'representational capacity of vector embeddings'
Top K: 5, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
than 1k dimensions, while the largest embeddings used in research are around 4096 (Zhang et al.,
2025). We see that even with a moderate margin, which is needed to handle noise from messy data or
quantization, the lower bounds in Table 1 can already b

e larger than what is used in practice.
Additional constraints on real-world models (such as needing to generalize, learn from gradient
descent, and use natural language and tokenization) will make the dimension required in practice
much higher. As Table 1 shows, even a small multiple of this lower bound would make the embedding
dimension requirement infeasible. This multiple seems well-founded, as we will show in the next
section from the best-case optimization setting (e.g. free embeddings).
4 EMPIRICALCONNECTION: BESTCASEOPTIMIZATION
Having established a theoretical limitation of embedding models based on their embedding dimension
d, we seek to show that this holds empirically also.

than 1k dimensions, while the largest embeddings used in research are around 4096 (Zhang et al.,
2025). We see that even with a moderate margin, which is needed to handle noise from messy data or
quantization, the lower bounds in Table 1 can already be larger than what is used in practice.
Additional co